<a href="https://colab.research.google.com/github/umairmuk/python/blob/main/A2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# STEP 1: Import Libraries
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import classification_report, confusion_matrix

print("✅ TensorFlow Version:", tf.__version__)

# STEP 2: Set Dataset Paths
# Ensure your folder structure is: animal_dataset/train/{cat, dog, ...} and animal_dataset/test/...
train_dir = 'animal_dataset/train'
test_dir = 'animal_dataset/test'

IMG_SIZE = (128, 128)
BATCH_SIZE = 32
NUM_CLASSES = 5

# STEP 3: Data Preprocessing & Augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical' # Categorical for Multi-Class
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

class_names = list(train_generator.class_indices.keys())
print("\nClasses Found:", class_names)

# STEP 4: Visualize Sample Images
plt.figure(figsize=(15, 10))
images, labels = next(train_generator)
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(images[i])
    class_idx = np.argmax(labels[i])
    plt.title(class_names[class_idx], fontsize=10)
    plt.axis('off')
plt.tight_layout()
plt.show()

# STEP 5: Build Deeper CNN Model
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(128, 128, 3)),
    BatchNormalization(),
    MaxPooling2D(2, 2),

    Conv2D(64, (3, 3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2, 2),

    Conv2D(128, (3, 3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2, 2),

    Conv2D(256, (3, 3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2, 2),

    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(128, activation='relu'),
    Dropout(0.3),

    Dense(NUM_CLASSES, activation='softmax') # Softmax for Multi-Class
])

model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

# STEP 6: Train the Model
EPOCHS = 20
history = model.fit(train_generator, epochs=EPOCHS, validation_data=test_generator)

# STEP 7: Plot Training History
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history.history['accuracy'], label='Train Accuracy', color='green')
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy', color='red')
axes[0].set_title('Animal Classification - Accuracy'); axes[0].legend()

axes[1].plot(history.history['loss'], label='Train Loss', color='green')
axes[1].plot(history.history['val_loss'], label='Val Loss', color='red')
axes[1].set_title('Animal Classification - Loss'); axes[1].legend()
plt.show()

# STEP 8: Evaluate & Confusion Matrix
predictions = model.predict(test_generator)
y_pred = np.argmax(predictions, axis=1)
y_true = test_generator.classes

print("\n📊 CLASSIFICATION REPORT:")
print(classification_report(y_true, y_pred, target_names=class_names))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='viridis', xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix - Animal Classification')
plt.show()

# STEP 9: Save Model
model.save('animal_classification_model.h5')
print("✅ Model saved successfully!")